# 📱 Detector de Celular — Imágenes Propias
### La solución definitiva: entrenas con TU cámara, TU cara, TU entorno

**¿Por qué imágenes propias?**
Los datasets públicos tienen personas en autos, oficinas, etc.
Tu modelo necesita aprender a reconocerte a TI en TU habitación.

**Cuántas imágenes necesitas:**
- Mínimo: 60 por clase (120 total)
- Recomendado: 100 por clase (200 total)
- Más no es mejor — la variedad importa más que la cantidad

**Estructura:**
1. Instalaciones e imports
2. Captura de imágenes con tu cámara
3. Verificación del dataset capturado
4. Entrenamiento
5. Evaluación
6. Detección en tiempo real con alerta sonora

## Celda 1 — Instalaciones e Imports

In [ ]:
import os, base64, time, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from IPython.display import display, HTML, clear_output
from google.colab.output import eval_js
from google.colab import output as colab_output
import json as json_lib

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, roc_curve
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

IMG_SIZE = 224
BATCH    = 16
CLASES   = ['sin_celular', 'con_celular']

BASE_DIR  = '/content/mi_dataset'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

for split in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for clase in CLASES:
        os.makedirs(os.path.join(split, clase), exist_ok=True)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("✅ Listo para capturar imágenes")

## Celda 2 — Captura de Imágenes con tu Cámara

**Instrucciones:**
1. Haz clic en **"📵 Capturar SIN celular"** unas 100 veces con diferentes poses
2. Luego haz clic en **"📱 Capturar CON celular"** unas 100 veces sosteniendo tu celular
3. Cuando termines presiona **"✅ Guardar y continuar"**

**Consejos para buenas capturas:**
- Varía la distancia a la cámara (cerca, lejos, medio)
- Mueve la cabeza (izquierda, derecha, arriba, abajo)
- Cambia la posición del celular (oreja, frente, abajo, lado)
- Captura con diferentes iluminaciones si puedes
- Pon objetos de fondo distintos

In [ ]:
JS_CAPTURA = """
async function capturarDataset() {
  const resultados = [];
  const contadores = {sin_celular: 0, con_celular: 0};

  // ── Construir interfaz ──────────────────────────────────────────
  const ov = document.createElement('div');
  ov.style.cssText = 'position:fixed;inset:0;background:#0f172a;z-index:9999;display:flex;flex-direction:column;align-items:center;justify-content:center;font-family:monospace;gap:14px;';

  const tit = document.createElement('div');
  tit.style.cssText = 'color:#94a3b8;font-size:12px;letter-spacing:3px;';
  tit.textContent = 'CAPTURA DE DATASET';

  const video = document.createElement('video');
  video.style.cssText = 'width:480px;height:360px;border-radius:12px;display:block;object-fit:cover;border:2px solid #1e293b;';
  video.autoplay = true; video.muted = true;

  const flash = document.createElement('div');
  flash.style.cssText = 'position:absolute;inset:0;background:white;opacity:0;pointer-events:none;border-radius:12px;transition:opacity .05s;';

  const vwrap = document.createElement('div');
  vwrap.style.cssText = 'position:relative;';
  vwrap.append(video, flash);

  // Contadores
  const cntRow = document.createElement('div');
  cntRow.style.cssText = 'display:flex;gap:24px;';

  const cntSin = document.createElement('div');
  cntSin.style.cssText = 'text-align:center;';
  cntSin.innerHTML = '<div style="font-size:11px;color:#4ade80;letter-spacing:1px;">SIN CELULAR</div><div id="cnt-sin" style="font-size:36px;font-weight:bold;color:#4ade80;">0</div>';

  const cntCon = document.createElement('div');
  cntCon.style.cssText = 'text-align:center;';
  cntCon.innerHTML = '<div style="font-size:11px;color:#f87171;letter-spacing:1px;">CON CELULAR</div><div id="cnt-con" style="font-size:36px;font-weight:bold;color:#f87171;">0</div>';

  cntRow.append(cntSin, cntCon);

  // Botones
  const btnRow = document.createElement('div');
  btnRow.style.cssText = 'display:flex;gap:12px;';

  const btnSin = document.createElement('button');
  btnSin.innerHTML = '📵 Capturar SIN celular';
  btnSin.style.cssText = 'padding:12px 20px;background:#166534;color:white;border:none;border-radius:8px;cursor:pointer;font-size:14px;font-weight:bold;font-family:monospace;';

  const btnCon = document.createElement('button');
  btnCon.innerHTML = '📱 Capturar CON celular';
  btnCon.style.cssText = 'padding:12px 20px;background:#991b1b;color:white;border:none;border-radius:8px;cursor:pointer;font-size:14px;font-weight:bold;font-family:monospace;';

  const btnFin = document.createElement('button');
  btnFin.innerHTML = '✅ Guardar y continuar';
  btnFin.style.cssText = 'padding:12px 20px;background:#1e3a5f;color:#93c5fd;border:1px solid #3b82f6;border-radius:8px;cursor:pointer;font-size:13px;font-family:monospace;';

  btnRow.append(btnSin, btnCon, btnFin);

  const tips = document.createElement('div');
  tips.style.cssText = 'font-size:11px;color:#334155;text-align:center;';
  tips.textContent = 'Varía poses, distancias y posiciones del celular';

  ov.append(tit, vwrap, cntRow, btnRow, tips);
  document.body.appendChild(ov);

  // Iniciar cámara
  const stream = await navigator.mediaDevices.getUserMedia({
    video: {width:640, height:480, facingMode:'user'}
  });
  video.srcObject = stream;
  await new Promise(r => video.onloadeddata = r);

  const canvas = document.createElement('canvas');
  canvas.width = 640; canvas.height = 480;
  const ctx = canvas.getContext('2d');

  function capturar(clase) {
    ctx.drawImage(video, 0, 0, 640, 480);
    flash.style.opacity = '0.7';
    setTimeout(() => flash.style.opacity = '0', 60);
    const data = canvas.toDataURL('image/jpeg', 0.85).split(',')[1];
    resultados.push({clase, data});
    contadores[clase]++;
    document.getElementById('cnt-sin').textContent = contadores.sin_celular;
    document.getElementById('cnt-con').textContent = contadores.con_celular;
  }

  btnSin.onclick = () => capturar('sin_celular');
  btnCon.onclick = () => capturar('con_celular');

  await new Promise(resolve => { btnFin.onclick = resolve; });

  stream.getTracks().forEach(t => t.stop());
  document.body.removeChild(ov);
  return JSON.stringify(resultados);
}
capturarDataset();
"""

print("📸 Abriendo cámara para captura de imágenes...")
print()
print("INSTRUCCIONES:")
print("  📵 Sin celular → 100 fotos variando poses y distancias")
print("  📱 Con celular → 100 fotos con celular en distintas posiciones")
print()

resultado = eval_js(JS_CAPTURA)
capturas  = json_lib.loads(resultado)

# Guardar imágenes dividiendo 70/15/15
contadores_clase = {c: 0 for c in CLASES}

for item in capturas:
    clase = item['clase']
    n     = contadores_clase[clase]

    # Split según posición
    total_clase = len([x for x in capturas if x['clase'] == clase])
    pct         = n / total_clase if total_clase > 0 else 0

    if pct < 0.70:
        split = 'train'
    elif pct < 0.85:
        split = 'val'
    else:
        split = 'test'

    img_pil = Image.open(BytesIO(base64.b64decode(item['data']))).convert('RGB')
    img_pil = img_pil.resize((IMG_SIZE, IMG_SIZE))
    ruta    = os.path.join(BASE_DIR, split, clase, f'{clase}_{n:04d}.jpg')
    img_pil.save(ruta, 'JPEG', quality=90)
    contadores_clase[clase] += 1

# Resumen
print("\n✅ Imágenes guardadas:")
for split in ['train', 'val', 'test']:
    for clase in CLASES:
        n = len(os.listdir(os.path.join(BASE_DIR, split, clase)))
        print(f"   {split}/{clase}: {n}")

## Celda 3 — Verificación del Dataset Capturado

In [ ]:
# Contar total por clase
sin_total = sum(len(os.listdir(os.path.join(BASE_DIR, s, 'sin_celular')))
                for s in ['train','val','test'])
con_total = sum(len(os.listdir(os.path.join(BASE_DIR, s, 'con_celular')))
                for s in ['train','val','test'])

print(f"📊 Total imágenes capturadas:")
print(f"   sin_celular: {sin_total}")
print(f"   con_celular: {con_total}")
print(f"   Total:       {sin_total + con_total}")

if sin_total < 60 or con_total < 60:
    print()
    print("⚠️  Tienes pocas imágenes. Recomendado: mínimo 60 por clase.")
    print("   Vuelve a ejecutar la Celda 2 para capturar más.")
else:
    print("\n✅ Suficientes imágenes para entrenar")

# Mostrar muestra visual
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Tus imágenes capturadas — Sin celular (arriba) | Con celular (abajo)',
             fontsize=12, fontweight='bold')

for row, clase in enumerate(CLASES):
    ruta   = os.path.join(TRAIN_DIR, clase)
    imgs   = os.listdir(ruta)[:5]
    color  = 'darkgreen' if clase == 'sin_celular' else 'crimson'
    for col, archivo in enumerate(imgs):
        img = Image.open(os.path.join(ruta, archivo))
        axes[row][col].imshow(img)
        axes[row][col].set_title(clase.replace('_',' '), fontsize=9, color=color)
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

## Celda 4 — Entrenamiento

In [ ]:
# Generadores
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.15,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='binary', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='binary', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='binary', shuffle=False
)

print(f"Clases: {train_gen.class_indices}")
print(f"Train:  {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")

# ── Modelo ──────────────────────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model   = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Verificar punto de partida neutro
preds_ini = [float(model.predict(np.random.rand(1,224,224,3).astype('float32'),
             verbose=0)[0][0]) for _ in range(10)]
print(f"\nPromedio inicial (debe ser ~0.5): {np.mean(preds_ini):.4f}")

# ── FASE 1 ───────────────────────────────────────────────────────────
pasos_train = max(1, train_gen.samples // BATCH)
pasos_val   = max(1, val_gen.samples   // BATCH)

cb = [
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("\n🏋️  FASE 1 — Entrenando cabeza...")
h1 = model.fit(
    train_gen, steps_per_epoch=pasos_train,
    validation_data=val_gen, validation_steps=pasos_val,
    epochs=15, callbacks=cb, verbose=1
)
acc1 = max(h1.history['val_accuracy'])
print(f"\n✅ Fase 1 — Val Accuracy: {acc1*100:.1f}%")

# ── FASE 2 Fine-Tuning ───────────────────────────────────────────────
print("\n🔧 FASE 2 — Fine-Tuning...")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

h2 = model.fit(
    train_gen, steps_per_epoch=pasos_train,
    validation_data=val_gen, validation_steps=pasos_val,
    epochs=15, callbacks=cb, verbose=1
)
acc2 = max(h2.history['val_accuracy'])
print(f"\n✅ Fase 2 — Val Accuracy: {acc2*100:.1f}%")

# Verificar sesgo
preds_fin = [float(model.predict(np.random.rand(1,224,224,3).astype('float32'),
             verbose=0)[0][0]) for _ in range(20)]
promedio  = np.mean(preds_fin)
print(f"\n🔍 Promedio con ruido: {promedio:.4f}")
print("✅ Sin sesgo" if 0.3 < promedio < 0.7 else f"⚠️  Sesgo: umbral sugerido = {promedio+0.15:.2f}")

model.save('/content/modelo_propio.keras')
print("\n💾 Modelo guardado")

## Celda 5 — Evaluación

In [ ]:
test_gen.reset()
loss, acc, auc = model.evaluate(test_gen, verbose=0)

test_gen.reset()
y_prob = model.predict(test_gen, steps=len(test_gen), verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_gen.classes[:len(y_pred)]

print("═" * 40)
print("       MÉTRICAS — TEST SET")
print("═" * 40)
print(f"  Accuracy:  {acc*100:.1f}%")
print(f"  AUC:       {auc:.4f}")
print("═" * 40)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Evaluación del modelo', fontsize=13)

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin celular','Con celular'],
            yticklabels=['Sin celular','Con celular'],
            ax=axes[0], annot_kws={'size':14})
axes[0].set_title('Matriz de confusión')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicho')

# Distribución de probabilidades
axes[1].hist(y_prob[y_true==0], bins=20, alpha=0.7,
             color='#4CAF50', label='Sin celular', density=True)
axes[1].hist(y_prob[y_true==1], bins=20, alpha=0.7,
             color='#F44336', label='Con celular', density=True)
axes[1].axvline(0.5, color='black', linestyle='--', label='Umbral 0.5')
axes[1].set_title('Distribución de probabilidades')
axes[1].set_xlabel('P(con celular)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Celda 6 — Detección en Tiempo Real con Alerta Sonora

- Marco **rojo + alerta sonora** cuando detecta celular
- Marco **verde + silencio** cuando no hay celular
- Cooldown de 3 segundos entre alertas para no ser molesto
- **Para detener:** ejecuta la Celda 7

In [ ]:
try:
    model.load_weights('/content/modelo_propio.keras')
    print("✅ Pesos cargados")
except:
    print("ℹ️  Usando pesos actuales")

# Ajustar umbral si el modelo tiene sesgo
# Si el promedio de ruido fue > 0.7, sube este valor
UMBRAL   = 0.5
INTERVALO = 0.5
COOLDOWN  = 3.0

JS_FRAME = """
new Promise(async (resolve) => {
  if (!window._camStream) {
    window._camStream = await navigator.mediaDevices.getUserMedia({
      video: {width:640, height:480, facingMode:'user'}
    });
    window._camVideo = document.createElement('video');
    window._camVideo.srcObject = window._camStream;
    window._camVideo.autoplay = true;
    window._camVideo.muted = true;
    await new Promise(r => window._camVideo.onloadeddata = r);
  }
  const c = document.createElement('canvas');
  c.width=640; c.height=480;
  c.getContext('2d').drawImage(window._camVideo,0,0);
  resolve(c.toDataURL('image/jpeg',0.8).split(',')[1]);
});
"""

JS_ALERTA = """
(() => {
  const ctx = new (window.AudioContext||window.webkitAudioContext)();
  [880,660].forEach((f,i)=>{
    const o=ctx.createOscillator(),g=ctx.createGain();
    o.connect(g);g.connect(ctx.destination);
    o.type='sine';o.frequency.value=f;
    g.gain.setValueAtTime(0,ctx.currentTime+i*.2);
    g.gain.linearRampToValueAtTime(.5,ctx.currentTime+i*.2+.02);
    g.gain.exponentialRampToValueAtTime(.001,ctx.currentTime+i*.2+.2);
    o.start(ctx.currentTime+i*.2);
    o.stop(ctx.currentTime+i*.2+.25);
  });
})()
"""

def mostrar(es_cel, pct, frames, n_cel, alarmas, historia, b64):
    color  = '#7f1d1d' if es_cel else '#14532d'
    bcolor = '#f87171' if es_cel else '#4ade80'
    txt    = '⚠  CELULAR DETECTADO' if es_cel else '✓  SIN CELULAR'
    pct_c  = round(n_cel/frames*100) if frames > 0 else 0
    bloqs  = ''.join([
        f'<div style="width:28px;height:18px;border-radius:3px;'
        f'background:{"#f87171" if h else "#4ade80"};'
        f'display:inline-block;margin:1px;"></div>'
        for h in historia
    ])
    clear_output(wait=True)
    display(HTML(f"""
    <div style="font-family:monospace;max-width:500px;">
      <img src="data:image/jpeg;base64,{b64}"
           style="width:400px;border-radius:10px;display:block;margin-bottom:10px;
                  border:3px solid {"#f87171" if es_cel else "#4ade80"};"/>
      <div style="font-size:20px;font-weight:bold;padding:12px;border-radius:8px;
                  background:{color};color:white;text-align:center;margin-bottom:8px;">
        {txt}
      </div>
      <div style="font-size:12px;color:#555;margin-bottom:4px;">Confianza: {pct}%</div>
      <div style="background:#e2e8f0;border-radius:6px;height:12px;margin-bottom:10px;">
        <div style="height:12px;border-radius:6px;width:{pct}%;background:{bcolor};"></div>
      </div>
      <div style="font-size:12px;color:#555;margin-bottom:8px;">
        Frames: {frames} | Con celular: {n_cel} ({pct_c}%) | Alertas: {alarmas}
      </div>
      <div style="margin-bottom:6px;">{bloqs}</div>
      <div style="font-size:11px;color:#aaa;">Para detener ejecuta la Celda 7</div>
    </div>
    """))

# ── Loop ─────────────────────────────────────────────────────────────
print("🎥 Iniciando... permite el acceso a la cámara\n")

frames=0; n_cel=0; alarmas=0
ultima=0.0; historia=[]; prev=None

try:
    while True:
        t0  = time.time()
        b64 = eval_js(JS_FRAME)

        img  = Image.open(BytesIO(base64.b64decode(b64))).convert('RGB')
        arr  = np.array(img.resize((IMG_SIZE,IMG_SIZE)))/255.0
        prob = float(model.predict(np.expand_dims(arr,0).astype('float32'),verbose=0)[0][0])
        es   = prob >= UMBRAL
        pct  = round(prob*100)
        now  = time.time()

        frames += 1
        if es: n_cel += 1

        if es and (( es and prev==False) or (now-ultima)>=COOLDOWN):
            eval_js(JS_ALERTA)
            ultima=now; alarmas+=1

        prev = es
        historia.append(es)
        if len(historia)>10: historia.pop(0)

        mostrar(es, pct, frames, n_cel, alarmas, historia, b64)

        time.sleep(max(0, INTERVALO-(time.time()-t0)))

except KeyboardInterrupt:
    pass
finally:
    eval_js("""
    if(window._camStream){
      window._camStream.getTracks().forEach(t=>t.stop());
      window._camStream=null;window._camVideo=null;
    }'ok'
    """)
    clear_output(wait=True)
    print(f"✅ Sesión finalizada — {frames} frames, {alarmas} alertas")

Reemplazo de la celda 6 por si llega a fallar

In [ ]:
try:
    model.load_weights('/content/modelo_propio.keras')
    print("✅ Pesos cargados")
except:
    print("ℹ️  Usando pesos actuales")

UMBRAL    = 0.5
INTERVALO = 0.5
COOLDOWN  = 3.0
IMG_SIZE  = 224

JS_INIT_CAM = """
new Promise(async (resolve) => {
  // Crear video visible en el output
  if (!window._camStream) {
    window._camStream = await navigator.mediaDevices.getUserMedia({
      video: {width:480, height:360, facingMode:'user'}
    });
  }
  let vid = document.getElementById('cam-video');
  if (!vid) {
    vid = document.createElement('video');
    vid.id = 'cam-video';
    vid.style.cssText = 'width:400px;height:300px;border-radius:10px;display:block;margin-bottom:10px;object-fit:cover;border:3px solid #22c55e;';
    vid.autoplay = true;
    vid.muted = true;
    vid.playsInline = true;
    document.getElementById('video-slot').appendChild(vid);
  }
  vid.srcObject = window._camStream;
  await new Promise(r => vid.onloadeddata = r);
  resolve('ok');
});
"""

JS_CAPTURAR = """
new Promise((resolve) => {
  const vid = document.getElementById('cam-video');
  const c = document.createElement('canvas');
  c.width=640; c.height=480;
  c.getContext('2d').drawImage(vid,0,0,640,480);
  resolve(c.toDataURL('image/jpeg',0.8).split(',')[1]);
});
"""

JS_ALERTA = """
(() => {
  const ctx = new (window.AudioContext||window.webkitAudioContext)();
  [880,660].forEach((f,i)=>{
    const o=ctx.createOscillator(),g=ctx.createGain();
    o.connect(g);g.connect(ctx.destination);
    o.type='sine';o.frequency.value=f;
    g.gain.setValueAtTime(0,ctx.currentTime+i*.2);
    g.gain.linearRampToValueAtTime(.5,ctx.currentTime+i*.2+.02);
    g.gain.exponentialRampToValueAtTime(.001,ctx.currentTime+i*.2+.2);
    o.start(ctx.currentTime+i*.2);
    o.stop(ctx.currentTime+i*.2+.25);
  });
})()
"""

# ── Crear interfaz UNA sola vez ──────────────────────────────────────
display(HTML("""
<div style="font-family:monospace;max-width:500px;">
  <button onclick="window._detener=true;this.textContent='Deteniendo...';this.style.background='#6b7280';"
    style="padding:10px 24px;background:#dc2626;color:white;border:none;
           border-radius:8px;cursor:pointer;font-size:13px;font-weight:bold;
           font-family:monospace;margin-bottom:12px;display:block;">
    DETENER DETECCIÓN
  </button>

  <div id="video-slot"></div>

  <div id="estado"
       style="font-size:20px;font-weight:bold;padding:12px;border-radius:8px;
              background:#14532d;color:white;text-align:center;margin-bottom:8px;">
    Iniciando cámara...
  </div>
  <div id="conf-label" style="font-size:12px;color:#555;margin-bottom:4px;">
    Confianza: --%
  </div>
  <div style="background:#e2e8f0;border-radius:6px;height:12px;margin-bottom:10px;">
    <div id="barra"
         style="height:12px;border-radius:6px;width:0%;background:#4ade80;
                transition:width .3s,background .3s;">
    </div>
  </div>
  <div id="stats" style="font-size:12px;color:#555;margin-bottom:8px;">
    Frames: 0 | Con celular: 0 (0%) | Alertas: 0
  </div>
  <div id="historial" style="margin-bottom:6px;"></div>
</div>
"""))

eval_js("window._detener = false; 'ok'")

# Iniciar video en vivo dentro del output
eval_js(JS_INIT_CAM)

def actualizar_ui(es_cel, pct, frames, n_cel, alarmas, historia):
    color_bg  = '#7f1d1d' if es_cel else '#14532d'
    color_bar = '#f87171' if es_cel else '#4ade80'
    color_brd = '#f87171' if es_cel else '#22c55e'
    txt       = '⚠  CELULAR DETECTADO' if es_cel else '✓  SIN CELULAR'
    pct_c     = round(n_cel/frames*100) if frames > 0 else 0
    bloqs = ''.join([
        f'<div style="width:28px;height:18px;border-radius:3px;'
        f'background:{"#f87171" if h else "#4ade80"};'
        f'display:inline-block;margin:1px;"></div>'
        for h in historia
    ])
    eval_js(f"""
    document.getElementById('cam-video').style.borderColor = '{color_brd}';
    document.getElementById('estado').textContent = '{txt}';
    document.getElementById('estado').style.background = '{color_bg}';
    document.getElementById('conf-label').textContent = 'Confianza: {pct}%';
    document.getElementById('barra').style.width = '{pct}%';
    document.getElementById('barra').style.background = '{color_bar}';
    document.getElementById('stats').textContent = 'Frames: {frames} | Con celular: {n_cel} ({pct_c}%) | Alertas: {alarmas}';
    document.getElementById('historial').innerHTML = `{bloqs}`;
    'ok'
    """)

# ── Loop ─────────────────────────────────────────────────────────────
print("🎥 Cámara activa — analizando en tiempo real")

frames=0; n_cel=0; alarmas=0
ultima=0.0; historia=[]; prev=None

while True:
    detener = eval_js("window._detener || false")
    if detener:
        break

    t0  = time.time()
    b64 = eval_js(JS_CAPTURAR)

    img  = Image.open(BytesIO(base64.b64decode(b64))).convert('RGB')
    arr  = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    prob = float(model.predict(
        np.expand_dims(arr, 0).astype('float32'), verbose=0)[0][0])
    es   = prob >= UMBRAL
    pct  = round(prob * 100)
    now  = time.time()

    frames += 1
    if es: n_cel += 1

    if es and ((es and prev == False) or (now - ultima) >= COOLDOWN):
        eval_js(JS_ALERTA)
        ultima = now
        alarmas += 1

    prev = es
    historia.append(es)
    if len(historia) > 10:
        historia.pop(0)

    actualizar_ui(es, pct, frames, n_cel, alarmas, historia)
    time.sleep(max(0, INTERVALO - (time.time() - t0)))

eval_js("""
if(window._camStream){
  window._camStream.getTracks().forEach(t=>t.stop());
  window._camStream=null; window._camVideo=null;
} 'ok'
""")
print(f"\n✅ Sesión finalizada — {frames} frames | {alarmas} alertas")

## Celda 7 — Detener la Cámara

In [ ]:
from google.colab.output import eval_js
eval_js("""
if(window._camStream){
  window._camStream.getTracks().forEach(t=>t.stop());
  window._camStream=null;window._camVideo=null;
}'ok'
""")
print("✅ Cámara detenida")